In [ ]:
"""
Preprocess and scale CIC_ToN_IoT (target domain). Reuse scaler from CICIDS2017,
and calculate covariance statistics.
"""

### Imports ###
import json
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

# Load shared feature-space artifacts in a single, validated format.
def load_feature_order(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    if isinstance(payload, dict):
        if "features" not in payload:
            raise ValueError(
                f"Expected key 'features' in {path} when JSON object is provided."
            )
        feature_order = list(payload["features"])
    elif isinstance(payload, list):
        feature_order = list(payload)
    else:
        raise ValueError(
            f"Unsupported shared feature space format in {path}: "
            f"{type(payload).__name__}"
        )

    if not feature_order:
        raise ValueError(f"Shared feature space in {path} is empty")

    return feature_order

In [ ]:
### Import CSV ###

# Creates a Path object pointing to the target-domain CSV directory.
data_dir = Path("data/raw/target")

# Read CSV with encoding fallback for files that are not UTF-8.
def read_csv_with_fallback(file_path):
    for enc in ("utf-8", "cp1252", "latin1"):
        try:
            return pd.read_csv(file_path, low_memory=False, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError("unknown", b"", 0, 1, f"Unable to decode {file_path}")

# Target domain is expected to be a single CSV file.
csv_files = sorted(data_dir.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {data_dir}")
if len(csv_files) != 1:
    raise ValueError(
        f"Expected exactly one target CSV in {data_dir}, found {len(csv_files)}: "
        f"{[p.name for p in csv_files]}"
    )

csv_path = csv_files[0]
df = read_csv_with_fallback(csv_path)

# Display a quick shape check and preview rows.
print(f"Loaded target CSV: {csv_path.name}")
print("Dataset shape:", df.shape)
df.head()

In [ ]:
### Data sanitization ###

# Handle missing values by replacing all occurrences of infinity with NaN
# then removing rows containing NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# Remove duplicate flows and irrelevant columns.
df.drop_duplicates(inplace=True)
df = df.drop(columns=["Flow ID", "Source IP", "Destination IP", "Timestamp"], errors="ignore")

# Remove leading/trailing spaces from all column names.
df.rename(columns=lambda x: x.strip(), inplace=True)

In [ ]:
### Feature-space alignment (align features according to predetermined shared feature space) ###

# Canonical feature-space contract shared by source and target pipelines.
FEATURE_LIST_PATH = Path("data/processed/shared_feature_space.json")

# Handle missing file
if not FEATURE_LIST_PATH.exists():
    raise FileNotFoundError(
        f"Shared feature list not found at {FEATURE_LIST_PATH}. "
        "Create/populate this artifact before running preprocessing."
    )

# Load canonical ordered features used by source preprocessing/model training.
shared_features = load_feature_order(FEATURE_LIST_PATH)

# Label column can vary slightly by export; detect robustly.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)

# Ensure contiguous row index before splitting/rejoining feature and label columns.
df = df.reset_index(drop=True)
feature_df = df.drop(columns=[label_col]).copy() if label_col else df.copy()

def align_feature_space(frame, feature_list, fill_missing=False, fill_value=0.0):
    # Validate target columns against the expected shared feature contract.
    feature_list = list(feature_list)
    incoming = set(frame.columns)
    expected = set(feature_list)

    extra = sorted(incoming - expected)
    missing = sorted(expected - incoming)

    # Strict mode (default): fail if contract features are absent.
    if missing and not fill_missing:
        preview = missing[:10]
        raise ValueError(
            f"Missing required features: {preview} (total={len(missing)})"
        )

    # Tolerant mode can be enabled later if needed for sparse target schemas.
    if missing and fill_missing:
        for col in missing:
            frame[col] = fill_value

    # Drop unexpected columns and enforce canonical order for scaler/model input.
    aligned = frame[feature_list].copy()
    return aligned, extra, missing

aligned_X, dropped_extra, missing_cols = align_feature_space(
    feature_df,
    shared_features,
    fill_missing=False,
    fill_value=0.0,
)

# Reattach labels by position (not index label) to avoid accidental NaNs.
if label_col:
    labels_aligned = df[[label_col]].reset_index(drop=True)
    aligned_X = aligned_X.reset_index(drop=True)
    if len(aligned_X) != len(labels_aligned):
        raise ValueError(
            f"Feature/label row count mismatch after alignment: "
            f"X={len(aligned_X)}, y={len(labels_aligned)}"
        )
    df = pd.concat([aligned_X, labels_aligned], axis=1)
    print(f"Missing labels after reattach: {int(df[label_col].isna().sum())}")
else:
    df = aligned_X

print(f"Loaded shared feature list from {FEATURE_LIST_PATH}")
print(f"Aligned target feature count: {len(shared_features)}")
print(f"Dropped extra columns: {len(dropped_extra)}")
print(f"Missing required columns: {len(missing_cols)}")

In [ ]:
### Label-space alignment (align labels according to predetermined shared label space) ###

SHARED_LABEL_SPACE_PATH = Path("data/processed/shared_label_space.json")
TARGET_LABEL_MAP_PATH = Path("data/processed/target_label_map.json")

if not SHARED_LABEL_SPACE_PATH.exists():
    raise FileNotFoundError(
        f"Shared label space file not found at {SHARED_LABEL_SPACE_PATH}"
    )
if not TARGET_LABEL_MAP_PATH.exists():
    raise FileNotFoundError(
        f"Target label map file not found at {TARGET_LABEL_MAP_PATH}"
    )

with open(SHARED_LABEL_SPACE_PATH, "r", encoding="utf-8") as f:
    shared_label_space = json.load(f)
with open(TARGET_LABEL_MAP_PATH, "r", encoding="utf-8") as f:
    target_label_map = json.load(f)

# Re-detect label column to keep this cell independently runnable.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

# Normalize raw labels for stable matching (trim spaces, keep missing as <NA>).
raw_labels = df[label_col].astype("string").str.strip()

# Detect genuinely missing labels (null/blank) separately from unmapped labels.
missing_label_mask = raw_labels.isna() | raw_labels.eq("")
missing_label_count = int(missing_label_mask.sum())

if missing_label_count > 0:
    missing_indices = df.index[missing_label_mask].tolist()
    preview_rows = missing_indices[:10]
    preview_values = [repr(v) for v in raw_labels.loc[preview_rows].tolist()]
    raise ValueError(
        f"Missing label values found at row indices {preview_rows} "
        f"(total={missing_label_count}). "
        f"Sample raw values at those rows: {preview_values}. "
        "Clean/drop these rows before label alignment."
    )

# Guardrail: mapping file must only map into allowed shared classes.
invalid_target_classes = sorted(
    set(target_label_map.values()) - set(shared_label_space)
)
if invalid_target_classes:
    raise ValueError(
        "target_label_map.json contains classes not present in shared_label_space.json: "
        f"{invalid_target_classes}"
    )

# Apply raw->shared mapping.
mapped_labels = raw_labels.map(target_label_map)

# Fail fast on unmapped non-missing raw labels to avoid silent label drift.
unmapped_mask = (~missing_label_mask) & mapped_labels.isna()
if unmapped_mask.any():
    unmapped_indices = df.index[unmapped_mask].tolist()
    unmapped_raw = raw_labels[unmapped_mask].tolist()
    unique_unmapped_raw = sorted(set(unmapped_raw))
    preview_pairs = list(zip(unmapped_indices, [repr(v) for v in unmapped_raw]))[:10]

    raise ValueError(
        f"Unmapped raw labels found: {[repr(v) for v in unique_unmapped_raw[:10]]} "
        f"(total unique={len(unique_unmapped_raw)}, total rows={len(unmapped_indices)}). "
        f"Sample row/value pairs: {preview_pairs}. "
        f"Update {TARGET_LABEL_MAP_PATH}."
    )

# Replace dataset labels with aligned shared classes.
df[label_col] = mapped_labels

In [ ]:
### Scaling (reuse scaler of source dataset) ###

SCALER_PATH = Path("models/source_scaler.joblib")
if not SCALER_PATH.exists():
    raise FileNotFoundError(f"Source scaler not found at {SCALER_PATH}")

scaler = joblib.load(SCALER_PATH)

# Re-detect label column to keep this cell independently runnable.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

X = df.drop(columns=[label_col]).copy()
X_scaled = scaler.transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

labels = df[[label_col]].reset_index(drop=True)
X_scaled_df = X_scaled_df.reset_index(drop=True)
df = pd.concat([X_scaled_df, labels], axis=1)

print(f"Loaded scaler from {SCALER_PATH}")
print(f"Scaled target dataset shape: {X_scaled_df.shape}")

In [ ]:
### Label encoding (reuse encoder of shared label space) ###

ENCODER_PATH = Path("models/label_encoder.joblib")
if not ENCODER_PATH.exists():
    raise FileNotFoundError(f"Label encoder not found at {ENCODER_PATH}")

le = joblib.load(ENCODER_PATH)

# Re-detect label column to keep this cell independently runnable.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

raw_labels = df[label_col].astype("string").str.strip()
unknown_labels = sorted(set(raw_labels.dropna().tolist()) - set(le.classes_))
if unknown_labels:
    raise ValueError(
        f"Found labels not present in fitted source encoder: {unknown_labels[:10]} "
        f"(total={len(unknown_labels)})."
    )

encoded_labels = le.transform(raw_labels)
df[label_col] = encoded_labels

print(f"Loaded label encoder from {ENCODER_PATH}")
print(f"Encoded classes ({len(le.classes_)}): {list(le.classes_)}")

In [ ]:
### Calculate and export covariance and mean statistics ###

# Load shared feature space contract.
shared_feature_space_path = Path("data/processed/shared_feature_space.json")
if not shared_feature_space_path.exists():
    raise FileNotFoundError(f"Shared feature space file not found at {shared_feature_space_path}")

# Reuse unified parser so feature-space JSON is handled consistently across cells.
feature_order = load_feature_order(shared_feature_space_path)

# Re-detect label column to keep this cell independently runnable.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)

# Ensure exact feature order before CORAL statistics.
X_target = df.drop(columns=[label_col]).copy() if label_col else df.copy()
X_target_aligned = X_target[feature_order]

# Convert to numpy for CORAL math.
X_trg = X_target_aligned.to_numpy(dtype=np.float32)

# 1) Feature-wise mean vector.
target_feature_mean = np.mean(X_trg, axis=0)

# 2) Centered target data.
X_trg_centered = X_trg - target_feature_mean

# 3) Covariance matrix (core CORAL statistic).
target_covariance = np.cov(X_trg_centered, rowvar=False)

# Sanity checks.
assert target_covariance.shape[0] == target_covariance.shape[1], "Covariance matrix must be square"
assert target_covariance.shape[0] == len(feature_order), "Covariance dimension mismatch with feature space"

# Package CORAL statistics.
coral_target_stats = {
    "feature_order": feature_order,
    "mean": target_feature_mean,
    "covariance": target_covariance,
}

# Persist for downstream domain adaptation pipeline.
coral_stats_path = Path("models/coral_target_stats.joblib")
coral_stats_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(coral_target_stats, coral_stats_path)

print("CORAL target statistics extracted and saved successfully.")
print(f"Saved to: {coral_stats_path}")
print(f"Features: {len(feature_order)}")
print(f"Covariance shape: {target_covariance.shape}")

In [ ]:
### Export processed data ###

# Create output directory for processed target data.
output_dir = Path("data/processed/target")
output_dir.mkdir(parents=True, exist_ok=True)

# Save processed target data (single file).
target_df = pd.DataFrame(df).copy()
target_df.to_csv(output_dir / "target.csv", index=False)

print("Saved dataset:")
print(f"  Target: {len(target_df)} samples")
print(f"  Output file: {output_dir / 'target.csv'}")